In [1]:
!pip install earthengine-api --upgrade


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: E:\PYTHON\python.exe -m pip install --upgrade pip


In [2]:
%pip install earthengine-api


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import ee

d:\anaconda\envs\dl_env\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [4]:

# Define your service account email and path to the JSON key
SA_EMAIL = 'shoreline-pipeline@gen-lang-client-0412358476.iam.gserviceaccount.com'
KEY_PATH = r'E:\PROKER KKN\gen-lang-client-0412358476-68a272c7e246.json'

# Pass credentials directly to the client
credentials = ee.ServiceAccountCredentials(SA_EMAIL, KEY_PATH)
ee.Initialize(credentials, project='gen-lang-client-0412358476')

# Verify the connection works
print(ee.Image("NASA/NASADEM_HGT/001").get("title").getInfo())


NASADEM: NASA NASADEM Digital Elevation 30m


## Test: getDownloadURL (permission earthengine.thumbnails.create)

Cell 8 di atas hanya membuktikan **metadata read** (role `viewer` sudah cukup).
Pipeline (fetch.py -> export.py) memanggil `getDownloadURL`, yang butuh role
`earthengine.resourceEditor` pada service account (IAM). Tambahkan role itu di
Google Cloud Console -> IAM & Admin -> IAM -> edit SA -> grant role
**Earth Engine Resource Editor** (roles/earthengine.resourceEditor), lalu jalankan
cell berikut untuk memverifikasi.

In [6]:
# Minimal test: reproduce the exact failing call (download_patch -> getDownloadURL)
# Expect before IAM fix:  Permission 'earthengine.thumbnails.create' denied
# Expect after IAM fix:   DOWNLOAD URL OK -> https://...
import ee

SA_EMAIL = 'shoreline-pipeline@gen-lang-client-0412358476.iam.gserviceaccount.com'
KEY_PATH = r'E:\PROKER KKN\gen-lang-client-0412358476-68a272c7e246.json'

credentials = ee.ServiceAccountCredentials(SA_EMAIL, KEY_PATH)
ee.Initialize(credentials, project='gen-lang-client-0412358476')

img = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").select("B8").first()
url = img.getDownloadURL({
    "region": ee.Geometry.Point([110.4666041, -5.7915568]).buffer(1280).bounds(),
    "scale": 10,
    "format": "NPY",
})
print("DOWNLOAD URL OK ->", url[:80])

DOWNLOAD URL OK -> https://earthengine.googleapis.com/v1/projects/gen-lang-client-0412358476/thumbn


## Test penuh: jalankan persis fungsi pipeline (download_patch)

Menggunakan modul pipeline yang sama (src.gee.composite + src.gee.export) pada
AOI pertama. Output yang diharapkan: `[Titik_XX] MNDWI patch OK -- shape (256, 256)`.

In [7]:
import sys, json
from datetime import datetime, timedelta

sys.path.insert(0, r"E:\PROKER KKN\shoreline-kemujan")
from src.gee.composite import fetch_sentinel_composite, get_union_bbox
from src.gee.export import download_patch

aois = json.load(open(r"E:\PROKER KKN\shoreline-kemujan\config\aoi_points.geojson"))["features"]
union_bbox = get_union_bbox(aois)

end = datetime.utcnow()
start = end - timedelta(days=120)
composite, n_scene = fetch_sentinel_composite(
    union_bbox, start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d"), 30)
print("composite scenes:", n_scene)

name, (lon, lat) = aois[0]["properties"]["name"], aois[0]["geometry"]["coordinates"]
region = ee.Geometry.Point([lon, lat]).buffer(256 * 10 / 2).bounds()
mndwi = download_patch(composite, "MNDWI", region, 10)
print(f"[{name}] MNDWI patch OK -- shape {mndwi.shape}")

composite scenes: 37
[Titik_02] MNDWI patch OK -- shape (257, 257)
